# 182 — Crypto / VASP / Cross-Border Investigation
Trains `models/182_model.pkl`. Checklist: plan.md §1. Heads: illicit-wallet detection (Elliptic, supervised), VASP-attribution (weak-label), cross-border routing.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path(os.getcwd()).resolve()
while not (ROOT / "lib").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import yaml
from pathlib import Path
cfg_path = ROOT / "config" / "run_config.yaml"
with open(cfg_path) as fh:
    RUN = yaml.safe_load(fh)
print("run config:", {k: RUN[k] for k in ("active_models", "input_source", "eval_mode")})
TH = RUN.get("thresholds", {})

In [ ]:
import lib.io_utils as io
import lib.artifacts as art
io.ensure_dirs()
print('models dir ->', io.MODELS_DIR)

## 1.1 Data ingestion
Loads the 7 case families, SNAP trust edges and the Elliptic AML trio.

In [ ]:
cases = io.load_182_cases()
for fam, recs in cases.items():
    print(f'{fam:24s} records={len(recs)}')
snap = io.load_snap_trust()
print('SNAP trust edges:', len(snap))
feat, classes, edgelist = io.load_elliptic(labeled_only=True)
print('Elliptic features rows:', len(feat), '| classes rows:', len(classes),
      '| edgelist rows:', len(edgelist))
print('labeled classes:', classes['class'].value_counts().to_dict())

## 1.2 / 1.3 Feature engineering & labels
Graph node features = Elliptic 166-d feature vectors; case-level features from the JSON families; VASP weak-labels mined from `vasp_responses` (flagged as weak in metadata).

## 1.4 / 1.5 Modeling & evaluation
Build and fit the multi-head model; metrics are stored on the object.

In [ ]:
import lib.model_182 as m182
model = m182.Model182().fit()
print('=== 182 metrics ===')
for head, m in model.metrics.items():
    print(head, {k: round(v, 4) if isinstance(v, float) else v for k, v in m.items()})

## 1.6 Output — serialize + smoke test
Save to `models/182_model.pkl` and emit one canonical payload to `182/OUT`.

In [ ]:
art.save_model(model, io.MODELS_DIR / '182_model.pkl',
                provenance={'model': '182', 'metrics': model.metrics})
print('saved', io.MODELS_DIR / '182_model.pkl')
import lib.schema as schema
rec = cases['cross_border_cases'][0]
payload = model.predict(rec, threshold=TH.get('alert', 0.7))
assert schema.is_valid(payload), 'contract violated'
out_path = io.write_out(182, payload, 'smoke', case_id=rec.get('sahyog_case_id'))
print('smoke-test wrote:', out_path.name)
print('validated against canonical contract: OK')